In [ ]:
# ============================================================
# HDLSS 5x5 CV (TSTR / TRTR / C2ST) on LUNG
# Logistic Regression ONLY
# Fold-wise BSTabDiff generator (train-only per fold)
# + Fidelity diagnostics (real vs synthetic, global generator)
# + BSTabDiff runtime + GPU/CPU usage (robust CUDA init + safe stats)
#
# Dataset:
#   csv_path="lung_combined_encoded.csv"
#   label_col="Label"
#   n = 203, d = 3312, 5 classes
#
# FORCE GPU: cuda:6 (falls back safely if not available/visible)
# ============================================================

import os
import time
import random
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.utils import check_random_state
from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import NearestNeighbors

# Optional CPU memory stats
try:
    import psutil
except Exception:
    psutil = None

# Optional: torch (for generator + GPU stats)
try:
    import torch
except Exception:
    torch = None


GLOBAL_SEED = 42
FORCE_GPU_IDX = 6  # <<< force cuda:6


# ============================================================
# 0) Robust CUDA selection + initialization + safe peak stats
# ============================================================
def safe_cuda_setup(forced_idx: int = 6):
    """
    Try to use cuda:{forced_idx}. If not available/visible, fall back to cuda:0 or CPU.
    NOTE: If CUDA_VISIBLE_DEVICES is set, indices are REMAPPED.
    Example: CUDA_VISIBLE_DEVICES="6" => cuda:0 is the physical GPU 6.
    """
    visible = os.environ.get("CUDA_VISIBLE_DEVICES", None)

    if torch is None or (not hasattr(torch, "cuda")):
        return 0, "cpu", False, 0, visible

    if not torch.cuda.is_available():
        return 0, "cpu", False, 0, visible

    n_devices = torch.cuda.device_count()
    if n_devices <= 0:
        return 0, "cpu", False, 0, visible

    gpu_idx = forced_idx if forced_idx < n_devices else 0

    try:
        torch.cuda.init()
        torch.cuda.set_device(gpu_idx)
        _ = torch.empty(1, device=f"cuda:{gpu_idx}")  # force context
        return gpu_idx, f"cuda:{gpu_idx}", True, n_devices, visible
    except Exception as e:
        print(f"[WARN] CUDA setup failed -> CPU fallback. {type(e).__name__}: {e}")
        return 0, "cpu", False, n_devices, visible


def safe_reset_peak_memory(cuda_ok: bool, gpu_idx: int):
    if not cuda_ok or torch is None or (not torch.cuda.is_available()):
        return
    try:
        torch.cuda.set_device(gpu_idx)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    except Exception as e:
        print(f"[WARN] Could not reset peak memory stats: {type(e).__name__}: {e}")


def safe_get_peak_allocated_bytes(cuda_ok: bool, gpu_idx: int) -> int:
    if not cuda_ok or torch is None or (not torch.cuda.is_available()):
        return 0
    try:
        torch.cuda.set_device(gpu_idx)
        return int(torch.cuda.max_memory_allocated())
    except Exception as e:
        print(f"[WARN] Could not read GPU peak memory: {type(e).__name__}: {e}")
        return 0


GPU_IDX, DEVICE_STR, CUDA_OK, CUDA_COUNT, CUDA_VISIBLE = safe_cuda_setup(FORCE_GPU_IDX)

if torch is not None:
    try:
        torch.set_float32_matmul_precision("medium")
    except Exception:
        pass

print("=== Device selection ===")
print(f"  CUDA_VISIBLE_DEVICES={CUDA_VISIBLE}")
print(f"  torch.cuda.is_available()={(torch is not None and torch.cuda.is_available())}")
print(f"  torch.cuda.device_count()={CUDA_COUNT}")
print(f"  Requested cuda:{FORCE_GPU_IDX} -> Using {DEVICE_STR} (CUDA_OK={CUDA_OK}, GPU_IDX={GPU_IDX})")


def set_seed(seed=GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None and CUDA_OK:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


# ============================================================
# 1) Logistic Regression ONLY (multi-class supported)
# ============================================================
def make_clf(seed=0):
    return Pipeline(
        [
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler(with_mean=True, with_std=True)),
            (
                "clf",
                LogisticRegression(
                    max_iter=10000,
                    class_weight="balanced",
                    solver="lbfgs",
                    n_jobs=None,
                    multi_class="auto",
                    random_state=seed,
                ),
            ),
        ]
    )


# ============================================================
# 2) 5x5 CV evaluation (fold-wise generator)
#    - Multi-class AUC: macro-OVR
# ============================================================
def eval_5x5_tstr_foldwise_gen(
    X_real,
    y_real,
    generator_fit_fn=None,
    n_syn=None,
    do_trtr=True,
    do_c2st=True,
    seed=0,
):
    if generator_fit_fn is None:
        raise ValueError("generator_fit_fn must be provided.")

    X_real = np.asarray(X_real)
    y_real = np.asarray(y_real).astype(int)

    rng = check_random_state(seed)
    rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=seed)

    tstr_auc, tstr_acc = [], []
    trtr_auc, trtr_acc = [], []
    c2st_auc, c2st_acc = [], []

    for fold_id, (tr, te) in enumerate(rkf.split(X_real, y_real), 1):
        Xtr, ytr = X_real[tr], y_real[tr]
        Xte, yte = X_real[te], y_real[te]

        fold_seed = int(rng.randint(0, 2**31 - 1))
        n_syn_fold = len(Xtr) if (n_syn is None) else int(n_syn)

        gen_fold = generator_fit_fn(Xtr, ytr, seed=fold_seed)
        X_syn_fold, R_syn_fold, y_syn_fold = gen_fold.sample(n=n_syn_fold)

        # ---------- TSTR ----------
        clf_tstr = make_clf(seed=fold_seed)
        clf_tstr.fit(X_syn_fold, y_syn_fold)
        proba_t_all = clf_tstr.predict_proba(Xte)

        # macro-OVR AUC for multi-class
        auc_t = roc_auc_score(yte, proba_t_all, multi_class="ovr", average="macro")
        pred_t = proba_t_all.argmax(axis=1)

        tstr_auc.append(auc_t)
        tstr_acc.append(accuracy_score(yte, pred_t))

        # ---------- TRTR ----------
        if do_trtr:
            clf_trtr = make_clf(seed=fold_seed)
            clf_trtr.fit(Xtr, ytr)
            proba_r_all = clf_trtr.predict_proba(Xte)
            auc_r = roc_auc_score(yte, proba_r_all, multi_class="ovr", average="macro")
            pred_r = proba_r_all.argmax(axis=1)
            trtr_auc.append(auc_r)
            trtr_acc.append(accuracy_score(yte, pred_r))

        # ---------- C2ST (binary discriminator: real vs synth) ----------
        if do_c2st:
            n_mix = min(len(Xtr), len(X_syn_fold))
            idx_r = rng.choice(len(Xtr), size=n_mix, replace=False)
            idx_s = rng.choice(len(X_syn_fold), size=n_mix, replace=False)

            Xmix = np.vstack([Xtr[idx_r], X_syn_fold[idx_s]])
            ymix = np.concatenate([np.ones(n_mix, dtype=int), np.zeros(n_mix, dtype=int)])
            p = rng.permutation(len(ymix))
            Xmix, ymix = Xmix[p], ymix[p]

            cut = int(0.8 * len(ymix))
            Xmix_tr, Xmix_te = Xmix[:cut], Xmix[cut:]
            ymix_tr, ymix_te = ymix[:cut], ymix[cut:]

            # LR discriminator (binary)
            clf_c2st = Pipeline(
                [
                    ("imp", SimpleImputer(strategy="median")),
                    ("sc", StandardScaler(with_mean=True, with_std=True)),
                    (
                        "clf",
                        LogisticRegression(
                            max_iter=10000,
                            class_weight="balanced",
                            solver="lbfgs",
                            n_jobs=None,
                            random_state=fold_seed,
                        ),
                    ),
                ]
            )

            clf_c2st.fit(Xmix_tr, ymix_tr)
            proba_m = clf_c2st.predict_proba(Xmix_te)[:, 1]
            pred_m = (proba_m >= 0.5).astype(int)

            c2st_auc.append(roc_auc_score(ymix_te, proba_m))
            c2st_acc.append(accuracy_score(ymix_te, pred_m))

        print(f"[Fold {fold_id:02d}] TSTR ACC={tstr_acc[-1]:.4f}, AUC={tstr_auc[-1]:.4f}")

    out = {
        "tstr_auc_mean": float(np.mean(tstr_auc)),
        "tstr_auc_std": float(np.std(tstr_auc)),
        "tstr_acc_mean": float(np.mean(tstr_acc)),
        "tstr_acc_std": float(np.std(tstr_acc)),
        "n_folds": len(tstr_auc),
        "trtr_auc_mean": float(np.mean(trtr_auc)) if do_trtr else None,
        "trtr_auc_std": float(np.std(trtr_auc)) if do_trtr else None,
        "trtr_acc_mean": float(np.mean(trtr_acc)) if do_trtr else None,
        "trtr_acc_std": float(np.std(trtr_acc)) if do_trtr else None,
        "c2st_auc_mean": float(np.mean(c2st_auc)) if do_c2st else None,
        "c2st_auc_std": float(np.std(c2st_auc)) if do_c2st else None,
        "c2st_acc_mean": float(np.mean(c2st_acc)) if do_c2st else None,
        "c2st_acc_std": float(np.std(c2st_acc)) if do_c2st else None,
    }
    return out


# ============================================================
# 3) CSV saving helpers
# ============================================================
def save_synth_csv(X_syn, y_syn, out_csv, feature_names=None):
    X_syn = np.asarray(X_syn)
    n, m = X_syn.shape
    if feature_names is None:
        feature_names = [f"f{j}" for j in range(m)]
    df = pd.DataFrame(X_syn, columns=feature_names)
    if y_syn is not None:
        df.insert(0, "y", np.asarray(y_syn).astype(int))
    df.to_csv(out_csv, index=False)
    print(f"Saved CSV: {out_csv} | shape={df.shape} | nan_frac={np.isnan(X_syn).mean():.6g}")
    return out_csv


def save_synth_mask_csv(R_syn, out_csv):
    R_syn = np.asarray(R_syn).astype(int)
    dfR = pd.DataFrame(R_syn, columns=[f"r{j}" for j in range(R_syn.shape[1])])
    dfR.to_csv(out_csv, index=False)
    print(f"Saved mask CSV: {out_csv} | shape={dfR.shape} | observed_frac={R_syn.mean():.6g}")
    return out_csv


# ============================================================
# 4) BSTabDiff generator: FIXED config
# ============================================================
from block_subunit_gen import FeatureSpec, fit_block_subunit_generator


def make_generator_fit_fn(
    M=64,  # LUNG: 3312 features -> moderate capacity
    prior_type="diffusion",
    device: str | None = None,
    prior_epochs=10000,
    prior_batch=128,
    prior_lr=1e-3,
    ema_decay=0.999,
    verbose_every=0,
):
    if device is None:
        device = DEVICE_STR  # uses cuda:6 if visible; otherwise fallback

    def _fit(Xtr, ytr, seed=0):
        feature_specs = [FeatureSpec(name=f"f{j}", kind="continuous") for j in range(Xtr.shape[1])]
        gen = fit_block_subunit_generator(
            X=Xtr,
            feature_specs=feature_specs,
            y=ytr,
            M=M,
            blocks=None,
            permute_features=False,
            prior_type=prior_type,
            device=device,
            seed=seed,
            prior_epochs=prior_epochs,
            prior_batch=prior_batch,
            prior_lr=prior_lr,
            verbose_every=verbose_every,
            save_dir=None,
            save_name="bstabdiff_lung",
            save_best=True,
            use_ema=True,
            ema_decay=ema_decay,
            return_train_info=False,
        )
        return gen

    return _fit


# ============================================================
# 5) Fidelity metrics (works for multi-class too)
# ============================================================
def _ks_and_wasserstein_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return 0.0, 0.0
    values = np.concatenate([x, y])
    labels = np.concatenate([np.zeros(nx, dtype=int), np.ones(ny, dtype=int)])
    order = np.argsort(values)
    v_sorted = values[order]
    labels_sorted = labels[order]
    c1 = (labels_sorted == 0).astype(float)
    c2 = 1.0 - c1
    cdf1 = np.cumsum(c1) / nx
    cdf2 = np.cumsum(c2) / ny
    diff = cdf1 - cdf2
    ks = float(np.max(np.abs(diff)))
    w1 = float(np.sum(np.abs(diff[:-1]) * np.diff(v_sorted))) if len(v_sorted) > 1 else 0.0
    return ks, w1


def _rank_transform(X):
    X = np.asarray(X, dtype=float)
    n, m = X.shape
    ranks = np.empty_like(X, dtype=float)
    for j in range(m):
        col = X[:, j]
        order = np.argsort(col)
        ranks[order, j] = np.arange(n, dtype=float)
    return ranks


def _moments_1d(X):
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    centered = X - mean
    var = (centered**2).mean(axis=0)
    eps = 1e-8
    std = np.sqrt(var + eps)
    skew = (centered**3).mean(axis=0) / (std**3)
    kurt = (centered**4).mean(axis=0) / (std**4)
    return mean, var, skew, kurt


def compute_fidelity_metrics(X_real, y_real, X_syn, y_syn, generator=None, random_state=0):
    X_real = np.asarray(X_real, dtype=float)
    X_syn = np.asarray(X_syn, dtype=float)

    nan_frac_real = float(np.isnan(X_real).mean())
    nan_frac_syn = float(np.isnan(X_syn).mean())

    X_real = np.nan_to_num(X_real, nan=0.0, posinf=0.0, neginf=0.0)
    X_syn = np.nan_to_num(X_syn, nan=0.0, posinf=0.0, neginf=0.0)

    n_features = X_real.shape[1]

    ks_stats = np.zeros(n_features, dtype=float)
    w1_stats = np.zeros(n_features, dtype=float)
    for j in range(n_features):
        ks_stats[j], w1_stats[j] = _ks_and_wasserstein_1d(X_real[:, j], X_syn[:, j])

    pearson_real = np.corrcoef(X_real, rowvar=False)
    pearson_syn = np.corrcoef(X_syn, rowvar=False)
    mask_offdiag = ~np.eye(n_features, dtype=bool)
    pearson_abs_diff = np.abs(pearson_real - pearson_syn)[mask_offdiag]

    ranks_real = _rank_transform(X_real)
    ranks_syn = _rank_transform(X_syn)
    spearman_real = np.corrcoef(ranks_real, rowvar=False)
    spearman_syn = np.corrcoef(ranks_syn, rowvar=False)
    spearman_abs_diff = np.abs(spearman_real - spearman_syn)[mask_offdiag]

    y_real = np.asarray(y_real).astype(int) if y_real is not None else None
    y_syn = np.asarray(y_syn).astype(int) if y_syn is not None else None

    mi_abs = None
    if y_real is not None and y_syn is not None:
        mi_real = mutual_info_classif(X_real, y_real, random_state=random_state)
        mi_syn = mutual_info_classif(X_syn, y_syn, random_state=random_state)
        mi_abs = np.abs(mi_real - mi_syn)

    mean_r, var_r, skew_r, kurt_r = _moments_1d(X_real)
    mean_s, var_s, skew_s, kurt_s = _moments_1d(X_syn)
    higher = {
        "mean_abs_diff": float(np.mean(np.abs(mean_r - mean_s))),
        "var_abs_diff": float(np.mean(np.abs(var_r - var_s))),
        "skew_abs_diff": float(np.mean(np.abs(skew_r - skew_s))),
        "kurt_abs_diff": float(np.mean(np.abs(kurt_r - kurt_s))),
    }

    ll_real = ll_syn = None
    if generator is not None:
        try:
            if hasattr(generator, "log_prob"):
                ll_real = float(np.mean(generator.log_prob(X_real, y_real)))
                ll_syn = float(np.mean(generator.log_prob(X_syn, y_syn)))
            elif hasattr(generator, "score_samples"):
                ll_real = float(np.mean(generator.score_samples(X_real)))
                ll_syn = float(np.mean(generator.score_samples(X_syn)))
        except Exception:
            ll_real = ll_syn = None

    nn_stats = {}
    try:
        nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
        nn.fit(X_real)
        dists, _ = nn.kneighbors(X_syn, return_distance=True)
        dists = dists.ravel()
        nn_stats = {
            "mean_nn_dist": float(np.mean(dists)),
            "min_nn_dist": float(np.min(dists)),
            "p1_nn_dist": float(np.percentile(dists, 1.0)),
            "p5_nn_dist": float(np.percentile(dists, 5.0)),
        }
    except Exception:
        nn_stats = {}

    metrics = {
        "ks_per_feature": ks_stats,
        "wasserstein_per_feature": w1_stats,
        "pearson_mean_abs_corr_diff": float(np.mean(pearson_abs_diff)),
        "pearson_max_abs_corr_diff": float(np.max(pearson_abs_diff)),
        "spearman_mean_abs_corr_diff": float(np.mean(spearman_abs_diff)),
        "spearman_max_abs_corr_diff": float(np.max(spearman_abs_diff)),
        "higher_order_moment_diffs": higher,
        "privacy_nn_stats": nn_stats,
        "nan_frac_real": nan_frac_real,
        "nan_frac_syn": nan_frac_syn,
    }

    if mi_abs is not None:
        metrics.update(
            {
                "mi_abs_diff": mi_abs,
                "mi_abs_diff_mean": float(np.mean(mi_abs)),
                "mi_abs_diff_max": float(np.max(mi_abs)),
            }
        )
    if ll_real is not None:
        metrics.update({"loglik_real_mean": ll_real, "loglik_syn_mean": ll_syn})

    return metrics


# ============================================================
# 6) Load LUNG dataset
# ============================================================
def load_lung_dataset(csv_path="lung_combined_encoded.csv", label_col="Label"):
    df = pd.read_csv(csv_path)
    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found.")

    y_raw = df[label_col].values
    X_df = df.drop(columns=[label_col]).select_dtypes(include=[np.number])
    X = X_df.to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, _ = pd.factorize(y_raw)
    else:
        unique = np.unique(y_raw)
        if not np.array_equal(unique, np.arange(unique.size)):
            y, _ = pd.factorize(y_raw)
        else:
            y = y_raw.astype(int, copy=False)

    return X, y, list(X_df.columns)


# ============================================================
# 7) Main
# ============================================================
if __name__ == "__main__":
    set_seed(GLOBAL_SEED)

    X, y, feature_names = load_lung_dataset(
        csv_path="lung_combined_encoded.csv",
        label_col="Label",
    )
    print(f"\nLoaded LUNG: X.shape={X.shape}, y.shape={y.shape}, classes={np.unique(y)}")

    generator_fit_fn = make_generator_fit_fn(
        M=64,
        prior_type="diffusion",
        device=DEVICE_STR,  # should be cuda:6 if visible; otherwise fallback
        prior_epochs=10000,
        prior_batch=128,
        prior_lr=1e-3,
        ema_decay=0.999,
        verbose_every=0,
    )

    # (A) Train generator once + resource usage
    proc = psutil.Process(os.getpid()) if psutil is not None else None
    safe_reset_peak_memory(CUDA_OK, GPU_IDX)

    t0 = time.time()
    gen_full = generator_fit_fn(X, y, seed=GLOBAL_SEED)
    gen_train_time = time.time() - t0

    cpu_after = proc.memory_info().rss if proc is not None else 0
    gpu_peak_bytes = safe_get_peak_allocated_bytes(CUDA_OK, GPU_IDX)

    print("\n=== BSTabDiff generator resources (trained once on LUNG) ===")
    print(f"  Train time: {gen_train_time:.2f} sec")
    print(f"  Peak GPU allocated: {gpu_peak_bytes / (1024**3):.3f} GiB")
    print(f"  CPU RSS at end:     {cpu_after / (1024**3):.3f} GiB")

    # Sample global synthetic dataset for fidelity
    N_SYN = X.shape[0]  # 203
    X_syn_global, R_syn_global, y_syn_global = gen_full.sample(n=N_SYN)
    save_synth_csv(X_syn_global, y_syn_global, out_csv=f"lung_synth_BSTabDiff_n{N_SYN}.csv", feature_names=feature_names)
    save_synth_mask_csv(R_syn_global, out_csv=f"lung_synth_BSTabDiff_n{N_SYN}_mask.csv")

    # (B) 5x5 CV TSTR/TRTR/C2ST
    print("\n=== 5x5 CV: STRICT fold-wise BSTabDiff + Logistic Regression (LUNG) ===")
    res = eval_5x5_tstr_foldwise_gen(
        X_real=X,
        y_real=y,
        generator_fit_fn=generator_fit_fn,
        n_syn=None,
        do_trtr=True,
        do_c2st=True,
        seed=0,
    )
    print(f"\n  TSTR ACC: {res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}")
    print(f"  TSTR AUC (macro-OVR): {res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}")
    print(f"  TRTR ACC: {res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}")
    print(f"  TRTR AUC (macro-OVR): {res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}")
    print(f"  C2ST ACC: {res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}")
    print(f"  C2ST AUC: {res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}")

    # (C) Fidelity
    fidelity = compute_fidelity_metrics(
        X_real=X,
        y_real=y,
        X_syn=X_syn_global,
        y_syn=y_syn_global,
        generator=gen_full,
        random_state=0,
    )

    print("\n=== LUNG Fidelity (real vs synthetic) ===")
    print(f"  KS: mean={np.mean(fidelity['ks_per_feature']):.4f}, max={np.max(fidelity['ks_per_feature']):.4f}")
    print(f"  W1: mean={np.mean(fidelity['wasserstein_per_feature']):.4f}, max={np.max(fidelity['wasserstein_per_feature']):.4f}")
    print(f"  Pearson |Δcorr|: mean={fidelity['pearson_mean_abs_corr_diff']:.4f}, max={fidelity['pearson_max_abs_corr_diff']:.4f}")
    print(f"  Spearman |Δcorr|: mean={fidelity['spearman_mean_abs_corr_diff']:.4f}, max={fidelity['spearman_max_abs_corr_diff']:.4f}")

    higher = fidelity["higher_order_moment_diffs"]
    print("  Higher-order moments (mean abs diff):")
    print(f"    mean: {higher['mean_abs_diff']:.4e}")
    print(f"    var:  {higher['var_abs_diff']:.4e}")
    print(f"    skew: {higher['skew_abs_diff']:.4e}")
    print(f"    kurt: {higher['kurt_abs_diff']:.4e}")

    if "mi_abs_diff_mean" in fidelity:
        print(f"  |ΔMI(feature,label)|: mean={fidelity['mi_abs_diff_mean']:.4f}, max={fidelity['mi_abs_diff_max']:.4f}")

    if fidelity["privacy_nn_stats"]:
        nn_s = fidelity["privacy_nn_stats"]
        print("  Privacy (NN dist synth -> nearest real):")
        print(f"    mean={nn_s['mean_nn_dist']:.4e}, min={nn_s['min_nn_dist']:.4e}, p1={nn_s['p1_nn_dist']:.4e}, p5={nn_s['p5_nn_dist']:.4e}")

=== Device selection ===
  CUDA_VISIBLE_DEVICES=None
  torch.cuda.is_available()=True
  torch.cuda.device_count()=8
  Requested cuda:6 -> Using cuda:6 (CUDA_OK=True, GPU_IDX=6)

Loaded LUNG: X.shape=(203, 3312), y.shape=(203,), classes=[0 1 2 3 4]

=== BSTabDiff generator resources (trained once on LUNG) ===
  Train time: 53.05 sec
  Peak GPU allocated: 0.031 GiB
  CPU RSS at end:     1.223 GiB
Saved CSV: lung_synth_BSTabDiff_n203.csv | shape=(203, 3313) | nan_frac=0.000113039
Saved mask CSV: lung_synth_BSTabDiff_n203_mask.csv | shape=(203, 3312) | observed_frac=0.999887

=== 5x5 CV: STRICT fold-wise BSTabDiff + Logistic Regression (LUNG) ===
[Fold 01] TSTR ACC=0.9756, AUC=0.9909
[Fold 02] TSTR ACC=0.9512, AUC=0.9981
[Fold 03] TSTR ACC=0.9756, AUC=1.0000
[Fold 04] TSTR ACC=0.9750, AUC=1.0000
[Fold 05] TSTR ACC=0.9250, AUC=0.9825
[Fold 06] TSTR ACC=0.9756, AUC=0.9954
[Fold 07] TSTR ACC=0.9756, AUC=0.9989
[Fold 08] TSTR ACC=0.9512, AUC=0.9861
[Fold 09] TSTR ACC=0.9750, AUC=0.9917
[Fold 1

**BSTabDiff w/ Lung Dataset (Logistic Regression as classifier)**